In [ ]:
from influxdb_client import InfluxDBClient, Point
from influxdb_client.client.write_api import SYNCHRONOUS
from datetime import datetime, timedelta, UTC
import random
import time
import math
import MySQLdb

# Config InfluxDB
URL = "http://influxdb2:8086"
TOKEN = "MyInitialAdminToken0=="
ORG = "docs"
BUCKET = "home"
MEASUREMENT = "home"

# Config MySQL
MYSQL_HOST = "mysql"
MYSQL_PORT = 3306
MYSQL_USER = "student"
MYSQL_PASSWORD = "student"
MYSQL_DATABASE = "home_data"

# Salles + Valeurs de base
ROOMS = {
    "LivingRoom": {
        "floor": "ground",
        "base_temp": 22.0,
        "base_hum": 40.0,
        "base_co2": 650,
        "base_power": 180,
    },
    "Kitchen": {
        "floor": "ground",
        "base_temp": 24.0,
        "base_hum": 45.0,
        "base_co2": 700,
        "base_power": 350,
    },
    "Bedroom": {
        "floor": "first",
        "base_temp": 20.5,
        "base_hum": 43.0,
        "base_co2": 600,
        "base_power": 90,
    },
    "Office": {
        "floor": "first",
        "base_temp": 21.5,
        "base_hum": 38.0,
        "base_co2": 750,
        "base_power": 250,
    },
    "Bathroom": {
        "floor": "first",
        "base_temp": 23.0,
        "base_hum": 60.0,
        "base_co2": 620,
        "base_power": 120,
    },
}

# Client InfluxDB
client = InfluxDBClient(url=URL, token=TOKEN, org=ORG)
write_api = client.write_api(write_options=SYNCHRONOUS)

# Client MySQL
mysql_conn = MySQLdb.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    passwd=MYSQL_PASSWORD,
    db=MYSQL_DATABASE,
    autocommit=False,
    connect_timeout=5,
)
mysql_cursor = mysql_conn.cursor()


INSERT_SQL = """
INSERT INTO sensor_data (
    time_ts, room, floor_name, temp, hum, co2, power, occupied
) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""


def generate_observation(room_name: str, config: dict, ts: datetime, live_variation: float = 1.0) -> dict:
    hour = ts.hour + ts.minute / 60.0

    daily_temp_variation = math.sin((hour / 24.0) * 2 * math.pi) * 1.5
    daily_hum_variation = math.cos((hour / 24.0) * 2 * math.pi) * 3.0

    occupied = False
    if room_name == "Office" and 9 <= ts.hour <= 18:
        occupied = random.random() < 0.75
    elif room_name == "LivingRoom" and 18 <= ts.hour <= 23:
        occupied = random.random() < 0.7
    elif room_name == "Kitchen" and ts.hour in [7, 12, 19]:
        occupied = random.random() < 0.8
    elif room_name == "Bedroom" and (ts.hour >= 22 or ts.hour <= 7):
        occupied = random.random() < 0.85
    elif room_name == "Bathroom" and ts.hour in [7, 8, 22]:
        occupied = random.random() < 0.5

    temp = config["base_temp"] + daily_temp_variation + random.uniform(-0.5, 0.5) * live_variation
    hum = config["base_hum"] + daily_hum_variation + random.uniform(-2.0, 2.0) * live_variation
    co2 = config["base_co2"] + random.randint(-40, 40)
    power = config["base_power"] + random.uniform(-20, 20) * live_variation

    if occupied:
        co2 += random.randint(150, 450)
        power += random.uniform(30, 120)

    if room_name == "Kitchen" and ts.hour in [12, 19]:
        temp += random.uniform(1.0, 2.5)
        hum += random.uniform(2.0, 5.0)
        power += random.uniform(150, 350)

    if room_name == "Bathroom" and ts.hour in [7, 8, 22]:
        hum += random.uniform(8.0, 15.0)

    if random.random() < 0.01:
        temp += random.uniform(2.0, 4.0)
    if random.random() < 0.01:
        co2 += random.randint(300, 700)

    temp = round(temp, 2)
    hum = round(max(20.0, min(hum, 90.0)), 2)
    co2 = int(max(350, co2))
    power = round(max(0.0, power), 2)

    return {
        "time_ts": ts,
        "room": room_name,
        "floor_name": config["floor"],
        "temp": temp,
        "hum": hum,
        "co2": co2,
        "power": power,
        "occupied": occupied,
    }


def observation_to_influx_point(obs: dict) -> Point:
    return (
        Point(MEASUREMENT)
        .tag("room", obs["room"])
        .tag("floor", obs["floor_name"])
        .field("temp", obs["temp"])
        .field("hum", obs["hum"])
        .field("co2", obs["co2"])
        .field("power", obs["power"])
        .field("occupied", obs["occupied"])
        .time(obs["time_ts"])
    )


def write_batch(observations: list[dict]) -> None:
    if not observations:
        return

    # InfluxDB
    influx_points = [observation_to_influx_point(obs) for obs in observations]
    write_api.write(bucket=BUCKET, org=ORG, record=influx_points)

    # MySQL
    mysql_rows = []
    for obs in observations:
        mysql_rows.append(
            (
                obs["time_ts"].replace(tzinfo=None),
                obs["room"],
                obs["floor_name"],
                obs["temp"],
                obs["hum"],
                obs["co2"],
                obs["power"],
                int(obs["occupied"]),
            )  
        )
    mysql_cursor.executemany(INSERT_SQL, mysql_rows)
    mysql_conn.commit()


def backfill_history(hours: int = 24, step_seconds: int = 10, batch_size: int = 2000) -> None:
    now = datetime.now(UTC)
    start = now - timedelta(hours=hours)
    observations = []

    current = start
    while current <= now:
        for room_name, config in ROOMS.items():
            observations.append(generate_observation(room_name, config, current, live_variation=0.8))

        if len(observations) >= batch_size:
            write_batch(observations)
            print(f"Backfill: {len(observations)} observations écrites dans InfluxDB + MySQL")
            observations.clear()

        current += timedelta(seconds=step_seconds)

    if observations:
        write_batch(observations)
        print(f"Backfill final: {len(observations)} observations écrites dans InfluxDB + MySQL")


def live_stream(delay_seconds: int = 1) -> None:
    try:
        while True:
            now = datetime.now(UTC)
            observations = []

            for room_name, config in ROOMS.items():
                observations.append(generate_observation(room_name, config, now, live_variation=1.2))

            write_batch(observations)

            summary = ", ".join(
                f'{obs["room"]}: temp={obs["temp"]}, hum={obs["hum"]}, co2={obs["co2"]}, power={obs["power"]}, occupied={obs["occupied"]}'
                for obs in observations
            )
            print(f"{now.isoformat()} -> {len(observations)} observations envoyées | {summary}")

            time.sleep(delay_seconds)

    except KeyboardInterrupt:
        print("Arrêt du script")


try:
    backfill_history(hours=24, step_seconds=10, batch_size=2000)
    mysql_cursor.execute("SELECT COUNT(*) FROM sensor_data;")
    print(mysql_cursor.fetchone())
    live_stream(delay_seconds=1)

finally:
    try:
        
        mysql_cursor.close()
        mysql_conn.close()
    except Exception:
        pass

    client.close()

Backfill: 2000 observations écrites dans InfluxDB + MySQL
Backfill: 2000 observations écrites dans InfluxDB + MySQL
Backfill: 2000 observations écrites dans InfluxDB + MySQL
Backfill: 2000 observations écrites dans InfluxDB + MySQL
Backfill: 2000 observations écrites dans InfluxDB + MySQL
Backfill: 2000 observations écrites dans InfluxDB + MySQL
Backfill: 2000 observations écrites dans InfluxDB + MySQL
Backfill: 2000 observations écrites dans InfluxDB + MySQL
Backfill: 2000 observations écrites dans InfluxDB + MySQL
Backfill: 2000 observations écrites dans InfluxDB + MySQL
Backfill: 2000 observations écrites dans InfluxDB + MySQL
Backfill: 2000 observations écrites dans InfluxDB + MySQL
Backfill: 2000 observations écrites dans InfluxDB + MySQL
Backfill: 2000 observations écrites dans InfluxDB + MySQL
Backfill: 2000 observations écrites dans InfluxDB + MySQL
Backfill: 2000 observations écrites dans InfluxDB + MySQL
Backfill: 2000 observations écrites dans InfluxDB + MySQL
Backfill: 2000